In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp01
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)




# ════════════════════════════════════════════════════════════════════════
# MLFP01 — Exercise 3: Functions and Aggregation
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this exercise, you will be able to:
#   - Define functions that accept parameters and return values
#   - Use loops to process collections and iterate over DataFrame rows
#   - Aggregate data by groups using group_by() + agg()
#   - Apply multiple statistics (mean, median, std, quantile) in one call
#   - Write reusable helper functions for common data analysis tasks
#
# PREREQUISITES: Complete Exercise 2 first (filtering, with_columns, chaining).
#
# ESTIMATED TIME: ~150-180 min
#
# TASKS:
#   1.  Write basic functions — def, parameters, return, type hints
#   2.  Functions with default parameters and docstrings
#   3.  Lists, dictionaries, and iteration patterns
#   4.  group_by() + agg() — the core aggregation pattern
#   5.  Multiple aggregations in one call — the full statistics toolkit
#   6.  Derived columns from aggregated results
#   7.  Multi-key group_by — cross-tabulation analysis
#   8.  Time-series aggregation — annual and quarterly trends
#   9.  Reusable analysis functions — parameterised district reports
#   10. Comprehensive ranked district report with for loop iteration
#
# DATASET: Singapore HDB resale flat transactions
#   Source: synthetic course dataset (hdb_resale.parquet) modelled on the
#   public HDB resale records — the numbers are not real market data
#   Rows: ~50,000 transactions | Columns: month, town, flat_type,
#   floor_area_sqm, resale_price, and more
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import polars as pl



### Data Loading


In [ ]:
loader = MLFPDataLoader()
hdb = loader.load("mlfp01", "hdb_resale.parquet")

# Add derived columns used throughout the exercise
hdb = hdb.with_columns(
    (pl.col("resale_price") / pl.col("floor_area_sqm")).alias("price_per_sqm"),
    pl.col("month").str.slice(0, 4).cast(pl.Int32).alias("year"),
    pl.col("month").str.slice(5, 2).cast(pl.Int32).alias("month_num"),
)

print("=" * 60)
print("  MLFP01 Exercise 3: Functions and Aggregation")
print("=" * 60)
print(f"\n  Data loaded: hdb_resale.parquet ({hdb.height:,} rows, {hdb.width} columns)")
print(f"  You're ready to start!\n")



## TASK 1: Writing basic functions — def, parameters, return


Args:
        amount: The numeric value to format.

    Returns:
        A string like "S$485,000".


Args:
        value: Decimal value (e.g., 0.85 for 85%).
        decimals: Number of decimal places (default 1).

    Returns:
        A string like "85.0%".


This uses if/elif/else — Python's branching mechanism. Each condition
    is checked in order; the first True branch executes and the rest are skipped.


In [ ]:
# A function packages reusable logic under a name. Without functions,
# you'd copy-paste the same code every time you need the same operation.
# def introduces the function, parameters go in parentheses,
# the body is indented, and return sends a value back to the caller.


def format_sgd(amount: float) -> str:
    # The : tells Python this is a type hint — it documents what type
    # the parameter should be, but Python does not enforce it at runtime.
    # -> str says this function returns a string.
    return f"S${amount:,.0f}"


def format_pct(value: float, decimals: int = 1) -> str:
    return f"{value * 100:.{decimals}f}%"


# price_range_label() uses if / elif / else to pick ONE branch. You'll
# study if/elif/else properly in Lesson 1.4 — for now, read it top to
# bottom: the first condition that is True decides the label.
def price_range_label(price: float) -> str:
    if price < 350_000:
        return "Budget (<350k)"
    elif price < 500_000:
        return "Mid-range (350k-500k)"
    elif price < 700_000:
        return "Premium (500k-700k)"
    else:
        return "Luxury (700k+)"


# --- Test the functions before using them in a pipeline ---
print("=== Function Tests ===")
print(f"format_sgd(485000):      {format_sgd(485_000)}")
print(f"format_sgd(1200000):     {format_sgd(1_200_000)}")
print(f"format_pct(0.85):        {format_pct(0.85)}")
print(f"format_pct(0.85, 2):     {format_pct(0.85, 2)}")
print(f"price_range_label(300k): {price_range_label(300_000)}")
print(f"price_range_label(485k): {price_range_label(485_000)}")
print(f"price_range_label(720k): {price_range_label(720_000)}")



### Checkpoint 1


In [ ]:
assert format_sgd(485_000) == "S$485,000", f"format_sgd returned: {format_sgd(485_000)}"
assert format_pct(0.5) == "50.0%", f"format_pct returned: {format_pct(0.5)}"
assert "Mid-range" in price_range_label(485_000), "485k should be Mid-range"
assert "Luxury" in price_range_label(720_000), "720k should be Luxury"
assert "Budget" in price_range_label(200_000), "200k should be Budget"
print("\n✓ Checkpoint 1 passed — basic functions work correctly\n")



## TASK 2: Functions with default parameters and validation


IQR measures spread without being skewed by extreme outliers.
    A wide IQR means prices vary a lot within the district.


CV normalises spread by the mean — useful for comparing variability
    between different towns or time periods regardless of absolute price level.


Returns a dict with all the key statistics — useful as a building
    block for more complex reports.


Safe division prevents ZeroDivisionError — common when computing ratios
    from data that might have zero counts in some categories.


In [ ]:
def compute_iqr(series: pl.Series) -> float:
    q75 = series.quantile(0.75)
    q25 = series.quantile(0.25)
    if q75 is None or q25 is None:
        return 0.0
    return q75 - q25


def compute_cv(series: pl.Series) -> float:
    mean = series.mean()
    std = series.std()
    if mean is None or std is None or mean == 0:
        return 0.0
    return std / mean * 100


def describe_series(series: pl.Series, name: str = "Series") -> dict:
    return {
        "name": name,
        "count": series.len(),
        "nulls": series.null_count(),
        "mean": series.mean(),
        "median": series.median(),
        "std": series.std(),
        "min": series.min(),
        "max": series.max(),
        "q25": series.quantile(0.25),
        "q75": series.quantile(0.75),
        "iqr": compute_iqr(series),
        "cv": compute_cv(series),
    }


def safe_divide(numerator: float, denominator: float, default: float = 0.0) -> float:
    if denominator == 0:
        return default
    return numerator / denominator


# --- Test the functions ---
test_prices = pl.Series("prices", [300_000, 400_000, 500_000, 600_000, 700_000])
print("=== Statistical Functions ===")
print(f"IQR:  {format_sgd(compute_iqr(test_prices))}")
print(f"CV:   {compute_cv(test_prices):.1f}%")

desc = describe_series(hdb["resale_price"], "Resale Price")
print(f"\n=== describe_series() output ===")
for key, val in desc.items():
    if isinstance(val, float) and val > 1000:
        print(f"  {key:>10}: {format_sgd(val)}")
    else:
        print(f"  {key:>10}: {val}")

print(f"\nSafe divide: 100/0 = {safe_divide(100, 0)}")
print(f"Safe divide: 100/3 = {safe_divide(100, 3):.2f}")



### Checkpoint 2


In [ ]:
assert (
    compute_iqr(test_prices) == 200_000.0
), f"IQR should be 200,000, got {compute_iqr(test_prices)}"
assert compute_cv(test_prices) > 0, "CV should be positive for non-constant series"
assert len(desc) >= 10, "describe_series should return at least 10 statistics"
assert desc["count"] == hdb.height, "count should match DataFrame height"
assert safe_divide(100, 0) == 0.0, "safe_divide by zero should return default"
print("\n✓ Checkpoint 2 passed — statistical helper functions work correctly\n")



## TASK 3: Lists, dictionaries, and iteration patterns


In [ ]:
# Before diving into group_by, let's solidify the iteration patterns
# you'll use to process aggregated results.

# --- 3a: Building a lookup dictionary from data ---
# Useful for mapping codes to human-readable names
flat_type_labels = {
    "1 ROOM": "Studio",
    "2 ROOM": "2-Room",
    "3 ROOM": "3-Room",
    "4 ROOM": "4-Room",
    "5 ROOM": "5-Room",
    "EXECUTIVE": "Executive",
    "MULTI-GENERATION": "Multi-Gen",
}

# --- 3b: Processing data with a for loop and accumulator ---
# Count transactions per flat type manually (to understand what group_by does)
flat_counts: dict[str, int] = {}
for ft in hdb["flat_type"].to_list():
    if ft in flat_counts:
        flat_counts[ft] += 1
    else:
        flat_counts[ft] = 1

print("=== Manual Flat Type Counts ===")
for ft, count in sorted(flat_counts.items(), key=lambda x: x[1], reverse=True):
    label = flat_type_labels.get(ft, ft)
    pct = count / hdb.height * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:<12} {count:>8,} ({pct:5.1f}%) {bar}")

# --- 3c: Zip and enumerate in practice ---
# zip() pairs two sequences element-by-element
# enumerate() adds a counter to each item
towns_sample = ["BISHAN", "QUEENSTOWN", "JURONG EAST", "WOODLANDS", "TAMPINES"]
town_medians = [
    hdb.filter(pl.col("town") == t)["resale_price"].median() for t in towns_sample
]

print(f"\n=== Sample Town Medians (zip + enumerate) ===")
for i, (town, median) in enumerate(zip(towns_sample, town_medians), start=1):
    print(f"  {i}. {town:<18} {format_sgd(median)}")

# --- 3d: List comprehensions with conditions ---
# Find towns where the median price exceeds S$500k
expensive_towns = [
    town
    for town, median in zip(towns_sample, town_medians)
    if median is not None and median > 500_000
]
print(f"\nTowns with median > S$500k: {expensive_towns}")

# --- 3e: Nested data structures ---
# A list of dicts — the pattern you'll see with .iter_rows(named=True)
town_profiles = [
    {
        "town": town,
        "median": median,
        "label": "premium" if median and median > 500_000 else "standard",
    }
    for town, median in zip(towns_sample, town_medians)
]

print(f"\n=== Town Profiles (list of dicts) ===")
for profile in town_profiles:
    print(
        f"  {profile['town']:<18} {format_sgd(profile['median'])}  [{profile['label']}]"
    )



### Checkpoint 3


In [ ]:
assert (
    len(flat_counts) == hdb["flat_type"].n_unique()
), "Manual count should match unique flat types"
assert sum(flat_counts.values()) == hdb.height, "Sum of counts should equal total rows"
assert len(town_medians) == len(towns_sample), "Should have one median per town"
assert len(expensive_towns) > 0, "Should have at least one expensive town"
print("\n✓ Checkpoint 3 passed — lists, dicts, and iteration patterns working\n")



## TASK 4: group_by() + agg() — the core aggregation pattern


In [ ]:
# group_by() splits the DataFrame into groups, one per unique value.
# agg() then computes a summary statistic for each group.
# The result has one row per group and one column per aggregation.
#
# This is the most important Polars pattern in data analysis:
# "For each [group], compute [statistics]."

# --- 4a: Basic group_by — one statistic per group ---
town_counts = (
    hdb.group_by("town")
    .agg(pl.len().alias("transaction_count"))
    .sort("transaction_count", descending=True)
)

print("=== Transaction Volume by Town ===")
print(f"Towns: {town_counts.height}")
print(town_counts.head(10))

# --- 4b: Flat type breakdown ---
flat_type_stats = (
    hdb.group_by("flat_type")
    .agg(
        pl.len().alias("count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("floor_area_sqm").median().alias("median_area"),
    )
    .sort("median_price")
)

print(f"\n=== Flat Type Summary ===")
for row in flat_type_stats.iter_rows(named=True):
    label = flat_type_labels.get(row["flat_type"], row["flat_type"])
    print(
        f"  {label:<12} {row['count']:>8,}  "
        f"median={format_sgd(row['median_price'])}  "
        f"area={row['median_area']:.0f}sqm"
    )
# INTERPRETATION: There's a clear price ladder from 2-room up to
# Multi-generation (this dataset has no 1-room flats).
# The median area also climbs — bigger flats cost more partly because
# they ARE more. The key question is: which flat types offer the best
# price per sqm? That answers "where is the value?"

# --- 4c: Compare group_by to manual counting ---
# Verify that group_by gives the same result as our manual count
manual_total = sum(flat_counts.values())
groupby_total = flat_type_stats["count"].sum()
print(f"\nManual total: {manual_total:,}  group_by total: {groupby_total:,}")
assert manual_total == groupby_total, "Totals should match"



### Checkpoint 4


In [ ]:
assert town_counts.height > 0, "town_counts should have rows"
assert town_counts.height == hdb["town"].n_unique(), "One row per unique town"
assert "transaction_count" in town_counts.columns, "Missing transaction_count column"
assert flat_type_stats.height == hdb["flat_type"].n_unique(), "One row per flat type"
print("\n✓ Checkpoint 4 passed — basic group_by/agg working correctly\n")



## TASK 5: Multiple aggregations — the full statistics toolkit


In [ ]:
district_stats = (
    hdb.group_by("town")
    .agg(
        # Volume
        pl.len().alias("transaction_count"),
        # Price statistics — multiple measures give a richer picture
        pl.col("resale_price").mean().alias("mean_price"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("resale_price").std().alias("std_price"),
        pl.col("resale_price").min().alias("min_price"),
        pl.col("resale_price").max().alias("max_price"),
        pl.col("resale_price").quantile(0.25).alias("q25_price"),
        pl.col("resale_price").quantile(0.75).alias("q75_price"),
        # Price per sqm — normalised measure for cross-size comparison
        pl.col("price_per_sqm").median().alias("median_price_sqm"),
        pl.col("price_per_sqm").mean().alias("mean_price_sqm"),
        # Area statistics
        pl.col("floor_area_sqm").median().alias("median_area_sqm"),
        pl.col("floor_area_sqm").mean().alias("mean_area_sqm"),
    )
    .sort("median_price", descending=True)
)

print(f"\n=== District Statistics (Top 10) ===")
print(
    district_stats.select(
        "town",
        "transaction_count",
        "median_price",
        "std_price",
        "median_price_sqm",
    ).head(10)
)

# --- Mean vs Median comparison ---
# When mean >> median, the distribution is right-skewed (expensive outliers)
print(f"\n=== Mean vs Median Price (Top 5) ===")
for row in district_stats.head(5).iter_rows(named=True):
    mean = row["mean_price"]
    median = row["median_price"]
    skew_indicator = "→ right-skewed" if mean > median * 1.05 else "→ roughly symmetric"
    print(
        f"  {row['town']:<20} mean={format_sgd(mean)}  "
        f"median={format_sgd(median)}  {skew_indicator}"
    )
# INTERPRETATION: Sorting by median_price shows the hierarchy of Singapore's
# housing market. When mean > median, expensive outliers pull the average up.
# For reporting, use median for "typical" price and mean for "total value."



### Checkpoint 5


In [ ]:
assert district_stats.height > 0, "district_stats should have rows"
assert "transaction_count" in district_stats.columns, "Missing transaction_count"
assert "median_price" in district_stats.columns, "Missing median_price"
top_median = district_stats["median_price"][0]
bottom_median = district_stats["median_price"][-1]
assert top_median >= bottom_median, "Results should be sorted descending"
print("\n✓ Checkpoint 5 passed — multiple aggregations producing correct statistics\n")



## TASK 6: Derived columns from aggregated results


In [ ]:
# Once you have aggregated stats, you can add further derived columns.

district_stats = district_stats.with_columns(
    # IQR: spread of the middle 50% of prices
    (pl.col("q75_price") - pl.col("q25_price")).alias("iqr_price"),
    # Coefficient of variation: std / mean * 100
    (pl.col("std_price") / pl.col("mean_price") * 100).alias("cv_price_pct"),
    # Premium ratio: what fraction of the max price is the median?
    (pl.col("median_price") / pl.col("max_price")).alias("premium_ratio"),
    # Price range: max - min — absolute spread
    (pl.col("max_price") - pl.col("min_price")).alias("price_range"),
    # Size-adjusted price premium: how much more than average does this town cost?
    (
        pl.col("median_price_sqm") / pl.col("median_price_sqm").mean().over(pl.lit(1))
    ).alias("price_index"),
)

# --- Market position classification ---
overall_median_psm = hdb["price_per_sqm"].median()

district_stats = district_stats.with_columns(
    pl.when(pl.col("median_price_sqm") > overall_median_psm * 1.15)
    .then(pl.lit("premium"))
    .when(pl.col("median_price_sqm") < overall_median_psm * 0.85)
    .then(pl.lit("value"))
    .otherwise(pl.lit("mainstream"))
    .alias("market_position"),
)

print(f"\n=== District Stats with Derived Columns ===")
print(
    district_stats.select(
        "town",
        "transaction_count",
        "median_price",
        "iqr_price",
        "cv_price_pct",
        "market_position",
    ).head(10)
)

# Market position summary
position_summary = (
    district_stats.group_by("market_position")
    .agg(
        pl.len().alias("towns"),
        pl.col("median_price").mean().alias("avg_median_price"),
        pl.col("cv_price_pct").mean().alias("avg_cv"),
    )
    .sort("avg_median_price", descending=True)
)
print(f"\n=== Market Position Summary ===")
print(position_summary)
# INTERPRETATION: cv_price_pct is key. Compare the CVs printed above:
# the lower a town's CV, the more tightly its prices cluster; a high CV
# means a wide mix — budget and premium side by side — or a few extreme
# outliers, so check the town's min/max before trusting it. Districts
# with high CV are harder to generalise about.



### Checkpoint 6


In [ ]:
assert "iqr_price" in district_stats.columns, "iqr_price should be added"
assert "cv_price_pct" in district_stats.columns, "cv_price_pct should be added"
assert "market_position" in district_stats.columns, "market_position should be added"
row = district_stats.row(0, named=True)
expected_iqr = row["q75_price"] - row["q25_price"]
assert abs(row["iqr_price"] - expected_iqr) < 1, "iqr should equal q75 - q25"
print("\n✓ Checkpoint 6 passed — derived columns computed from aggregates correctly\n")



## TASK 7: Multi-key group_by — cross-tabulation analysis


In [ ]:
# group_by() accepts multiple columns — creating a group for every
# unique combination of values.

# --- 7a: Town x Flat Type cross-tab ---
town_flat_stats = (
    hdb.group_by("town", "flat_type")
    .agg(
        pl.len().alias("count"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("price_per_sqm").median().alias("median_price_sqm"),
    )
    .sort("town", "flat_type")
)

print(f"\n=== Town x Flat Type Statistics ===")
print(f"Groups: {town_flat_stats.height}")
print(f"\nAng Mo Kio breakdown:")
print(town_flat_stats.filter(pl.col("town") == "ANG MO KIO"))

# --- 7b: Which town has the best value for 4-room flats? ---
four_room_ranking = (
    town_flat_stats.filter(pl.col("flat_type") == "4 ROOM")
    .filter(pl.col("count") >= 100)  # Only towns with enough data
    .sort("median_price_sqm")
)

print(f"\n=== Best Value Towns for 4-Room Flats (by price/sqm) ===")
print(f"{'Town':<22} {'Count':>7} {'Median Price':>14} {'Price/sqm':>12}")
print(f"{'─' * 57}")
for row in four_room_ranking.head(10).iter_rows(named=True):
    print(
        f"{row['town']:<22} {row['count']:>7,} "
        f"{format_sgd(row['median_price']):>14} "
        f"{format_sgd(row['median_price_sqm']):>12}"
    )
# INTERPRETATION: Sorting by price/sqm ascending shows where you get the
# most space for your money. Check whether the towns at the top of the
# list are away from the city centre — if so, the usual trade-off is
# commute time and amenity access.

# --- 7c: Flat type mix by town ---
# What percentage of each town's transactions are 4-room vs 5-room?
town_totals = town_flat_stats.group_by("town").agg(pl.col("count").sum().alias("total"))

flat_mix = (
    town_flat_stats.join(town_totals, on="town")
    .with_columns((pl.col("count") / pl.col("total") * 100).alias("pct_of_town"))
    .filter(pl.col("flat_type").is_in(["4 ROOM", "5 ROOM"]))
    .sort("town", "flat_type")
)

print(f"\n=== 4/5-Room Mix for Sample Towns ===")
for town in ["BISHAN", "TAMPINES", "WOODLANDS"]:
    town_data = flat_mix.filter(pl.col("town") == town)
    print(f"\n  {town}:")
    for row in town_data.iter_rows(named=True):
        print(
            f"    {row['flat_type']}: {row['pct_of_town']:.1f}% ({row['count']:,} txns)"
        )



### Checkpoint 7


In [ ]:
n_unique_pairs = hdb.select(["town", "flat_type"]).unique().height
assert (
    town_flat_stats.height == n_unique_pairs
), f"Expected {n_unique_pairs} groups, got {town_flat_stats.height}"
assert four_room_ranking.height > 0, "Should have 4-room rankings"
print("\n✓ Checkpoint 7 passed — multi-key group_by producing correct cross-tabs\n")



## TASK 8: Time-series aggregation — annual and quarterly trends


In [ ]:
# --- 8a: Annual transaction volume and price trend ---
annual_stats = (
    hdb.group_by("year")
    .agg(
        pl.len().alias("transactions"),
        pl.col("resale_price").median().alias("median_price"),
        pl.col("resale_price").mean().alias("mean_price"),
        pl.col("price_per_sqm").median().alias("median_psm"),
    )
    .sort("year")
)

print(f"\n=== Annual HDB Market Summary ===")
prev_price = None
for row in annual_stats.iter_rows(named=True):
    price = row["median_price"]
    yoy = ""
    if prev_price is not None and prev_price > 0:
        change = (price - prev_price) / prev_price * 100
        arrow = "↑" if change > 0 else "↓"
        yoy = f"  {arrow} {change:+.1f}%"
    print(
        f"  {row['year']}: {format_sgd(price):>14}  "
        f"psm={format_sgd(row['median_psm']):>10}  "
        f"vol={row['transactions']:>7,}{yoy}"
    )
    prev_price = price

# --- 8b: Annual stats by town — which towns grew fastest? ---
annual_by_town = (
    hdb.group_by("year", "town")
    .agg(
        pl.col("resale_price").median().alias("median_price"),
        pl.len().alias("transactions"),
    )
    .sort("year", "town")
)

# Compare 2020 to latest year for each town
years_available = sorted(hdb["year"].unique().to_list())
early_year = 2020 if 2020 in years_available else years_available[0]
late_year = years_available[-1]

early = annual_by_town.filter(pl.col("year") == early_year)
late = annual_by_town.filter(pl.col("year") == late_year)

growth = (
    early.select("town", pl.col("median_price").alias("price_early"))
    .join(
        late.select("town", pl.col("median_price").alias("price_late")),
        on="town",
    )
    .with_columns(
        (
            (pl.col("price_late") - pl.col("price_early")) / pl.col("price_early") * 100
        ).alias("growth_pct")
    )
    .sort("growth_pct", descending=True)
)

print(f"\n=== Price Growth {early_year} → {late_year} (Top 10 Towns) ===")
for row in growth.head(10).iter_rows(named=True):
    print(
        f"  {row['town']:<22} "
        f"{format_sgd(row['price_early'])} → {format_sgd(row['price_late'])}  "
        f"({row['growth_pct']:+.1f}%)"
    )

# --- 8c: Quarterly seasonality ---
quarter_map = {
    1: "Q1",
    2: "Q1",
    3: "Q1",
    4: "Q2",
    5: "Q2",
    6: "Q2",
    7: "Q3",
    8: "Q3",
    9: "Q3",
    10: "Q4",
    11: "Q4",
    12: "Q4",
}

hdb_q = hdb.with_columns(
    pl.col("month_num").replace_strict(quarter_map, default="Q?").alias("quarter")
)

# pl.len() counts every sale in the quarter across ALL years (pooled),
# so this is a total, not an average per year.
quarterly_volume = (
    hdb_q.group_by("quarter").agg(pl.len().alias("total_transactions")).sort("quarter")
)
print(f"\n=== Transactions by Quarter (all years pooled) ===")
print(quarterly_volume)
# INTERPRETATION: If Q1 had clearly fewer transactions than the other
# quarters, a Chinese New Year slowdown would be a plausible reason. Check
# the totals above: if the quarters are nearly equal, there is no such
# seasonal effect in this data.



### Checkpoint 8


In [ ]:
assert annual_stats.height > 0, "Should have annual stats"
assert growth.height > 0, "Should have growth data"
assert "growth_pct" in growth.columns, "growth_pct should exist"
print("\n✓ Checkpoint 8 passed — time-series aggregation working correctly\n")



## TASK 9: Reusable analysis functions — parameterised reports


Args:
        data: Full HDB DataFrame.
        town_name: Town to analyse (must match data values exactly).

    Returns:
        Dictionary with all computed statistics.


Pretty-print a town profile dictionary.


In [ ]:
# The real power of functions appears when you parameterise your analysis.
# Instead of copying code for each town, write one function and call it
# with different arguments.


def town_deep_dive(data: pl.DataFrame, town_name: str) -> dict:
    town_data = data.filter(pl.col("town") == town_name)

    if town_data.height == 0:
        return {"town": town_name, "error": "No data found"}

    prices = town_data["resale_price"]
    psm = town_data["price_per_sqm"]

    # Flat type breakdown
    flat_breakdown = (
        town_data.group_by("flat_type")
        .agg(pl.len().alias("count"), pl.col("resale_price").median().alias("median"))
        .sort("count", descending=True)
    )

    return {
        "town": town_name,
        "transactions": town_data.height,
        "pct_of_market": town_data.height / data.height * 100,
        "median_price": prices.median(),
        "mean_price": prices.mean(),
        "std_price": prices.std(),
        "iqr_price": compute_iqr(prices),
        "cv_pct": compute_cv(prices),
        "median_psm": psm.median(),
        "min_price": prices.min(),
        "max_price": prices.max(),
        "top_flat_type": flat_breakdown["flat_type"][0],
        "top_flat_count": flat_breakdown["count"][0],
        "years_active": town_data["year"].n_unique(),
    }


def print_town_profile(profile: dict) -> None:
    if "error" in profile:
        print(f"  {profile['town']}: {profile['error']}")
        return

    print(f"\n  ┌{'─' * 50}┐")
    print(f"  │ {profile['town']:^48} │")
    print(f"  ├{'─' * 50}┤")
    print(
        f"  │  Transactions:  {profile['transactions']:>8,} ({profile['pct_of_market']:.1f}% of market) │"
    )
    print(
        f"  │  Median Price:  {format_sgd(profile['median_price']):>12}                  │"
    )
    print(
        f"  │  Mean Price:    {format_sgd(profile['mean_price']):>12}                  │"
    )
    print(
        f"  │  Price Range:   {format_sgd(profile['min_price'])} - {format_sgd(profile['max_price'])}     │"
    )
    print(
        f"  │  IQR:           {format_sgd(profile['iqr_price']):>12}                  │"
    )
    print(f"  │  CV:            {profile['cv_pct']:>8.1f}%                        │")
    print(
        f"  │  Median PSM:    {format_sgd(profile['median_psm']):>12}                  │"
    )
    print(
        f"  │  Top Flat Type: {profile['top_flat_type']:<12} ({profile['top_flat_count']:,})      │"
    )
    print(f"  │  Years Active:  {profile['years_active']:>8}                        │")
    print(f"  └{'─' * 50}┘")


# --- Run deep dives for several towns ---
target_towns = ["BISHAN", "QUEENSTOWN", "TAMPINES", "WOODLANDS", "BUKIT MERAH"]
profiles = [town_deep_dive(hdb, town) for town in target_towns]

print("=== Town Deep Dives ===")
for profile in profiles:
    print_town_profile(profile)

# --- Compare towns side by side ---
print(f"\n=== Town Comparison ===")
print(f"{'Town':<18} {'Median':>12} {'PSM':>10} {'CV%':>6} {'Volume':>8}")
print(f"{'─' * 56}")
for p in sorted(profiles, key=lambda x: x.get("median_psm", 0), reverse=True):
    if "error" not in p:
        print(
            f"{p['town']:<18} {format_sgd(p['median_price']):>12} "
            f"{format_sgd(p['median_psm']):>10} "
            f"{p['cv_pct']:>5.1f}% "
            f"{p['transactions']:>8,}"
        )



### Checkpoint 9


In [ ]:
assert len(profiles) == len(target_towns), "Should have one profile per town"
assert all("town" in p for p in profiles), "Each profile should have a town key"
bishan_profile = next(p for p in profiles if p["town"] == "BISHAN")
assert bishan_profile["transactions"] > 0, "Bishan should have transactions"
assert bishan_profile["median_price"] > 0, "Bishan median should be positive"
print("\n✓ Checkpoint 9 passed — reusable analysis functions working correctly\n")



## TASK 10: Comprehensive ranked district report


Format one district row as a human-readable report line.


In [ ]:
# Combine everything into a publication-quality district ranking report.


def district_report_line(row: dict) -> str:
    town = row["town"]
    median = format_sgd(row["median_price"])
    count = row["transaction_count"]
    cv = row["cv_price_pct"]
    sqm = format_sgd(row["median_price_sqm"])
    position = row.get("market_position", "?")
    return (
        f"  {town:<22} {median:>12}  {count:>8,}  "
        f"CV={cv:5.1f}%  {sqm:>10}/sqm  [{position}]"
    )


# --- Full ranked report ---
print(f"\n{'═' * 78}")
print(f"  SINGAPORE HDB DISTRICT PRICE REPORT (All Years)")
print(f"{'═' * 78}")
print(
    f"  {'Town':<22} {'Median':>12}  {'Txns':>8}  "
    f"{'Spread':>8}  {'Per sqm':>10}  {'Position'}"
)
print(f"  {'─' * 74}")

for row in district_stats.iter_rows(named=True):
    print(district_report_line(row))

print(f"{'═' * 78}")

# --- Summary statistics across all districts ---
all_medians = district_stats["median_price"]
all_cvs = district_stats["cv_price_pct"]
all_counts = district_stats["transaction_count"]

print(f"\n=== Cross-District Summary ===")
print(f"  Total districts:           {district_stats.height}")
print(f"  Total transactions:        {all_counts.sum():,}")
print(f"  Most expensive district:   {format_sgd(all_medians.max())}")
print(f"  Least expensive district:  {format_sgd(all_medians.min())}")
print(f"  Average district median:   {format_sgd(all_medians.mean())}")
print(
    f"  Price gap (max - min):     {format_sgd(all_medians.max() - all_medians.min())}"
)
print(f"  Average CV:                {all_cvs.mean():.1f}%")
print(
    f"  Most active district:      {district_stats.sort('transaction_count', descending=True)['town'][0]}"
)
print(
    f"  Least active district:     {district_stats.sort('transaction_count')['town'][0]}"
)

# --- Market position breakdown ---
position_counts = (
    district_stats.group_by("market_position")
    .agg(
        pl.len().alias("districts"),
        pl.col("transaction_count").sum().alias("total_transactions"),
        pl.col("median_price").mean().alias("avg_median"),
    )
    .sort("avg_median", descending=True)
)

print(f"\n=== Market Position Breakdown ===")
for row in position_counts.iter_rows(named=True):
    print(
        f"  {row['market_position']:<12} "
        f"{row['districts']:>3} districts  "
        f"{row['total_transactions']:>8,} txns  "
        f"avg median={format_sgd(row['avg_median'])}"
    )

# INTERPRETATION: The spread between the most and least expensive districts
# quantifies Singapore's housing inequality. A wide spread (>S$200k) indicates
# that geography still matters significantly — where you buy affects your
# long-term asset value, not just your commute.



### Checkpoint 10


In [ ]:
assert all_medians.max() > all_medians.min(), "Most expensive > least expensive"
assert all_counts.sum() == hdb.height, "Total transactions should match dataset"
assert position_counts.height > 0, "Should have market position breakdown"
print("\n✓ Checkpoint 10 passed — comprehensive report generated correctly\n")



## REFLECTION


✓ Functions: def, parameters, return, type hints, docstrings
  ✓ Default parameters: optional arguments with sensible defaults
  ✓ Conditional logic: if/elif/else inside functions
  ✓ Statistical helpers: IQR, CV, safe_divide, describe_series
  ✓ Lists and dicts: comprehensions, zip, enumerate, nested structures
  ✓ group_by() + agg(): the core pattern for grouped statistics
  ✓ Multiple aggregations: mean, median, std, min, max, quantile
  ✓ Derived columns: IQR, CV, market position from aggregated data
  ✓ Multi-key grouping: group by (town, flat_type) simultaneously
  ✓ Time-series aggregation: annual trends, YoY growth, quarterly patterns
  ✓ Reusable functions: town_deep_dive() — parameterised analysis
  ✓ for loops: .iter_rows(named=True) to process each row as a dict
  ✓ Report building: combining statistics into formatted output

  NEXT: In Exercise 4, you'll combine data from multiple tables
  using joins — enriching HDB transactions with MRT station and
  school data. You'll learn when to use left vs inner joins, how to
  handle NULLs after a join, and why you must check what a column
  really measures before you join it.


In [ ]:
print("═" * 60)
print("  WHAT YOU'VE MASTERED")
print("═" * 60)
print(
)

